<a href="https://colab.research.google.com/github/SongXP111/Makemore/blob/main/makemore_part2_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [575]:
# Prepare necessary files
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2026-04-12 05:52:54--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt.22’

names.txt.22        100%[===================>] 222.80K  --.-KB/s    in 0.03s   

2026-04-12 05:52:54 (8.49 MB/s) - ‘names.txt.22’ saved [228145/228145]



In [576]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [577]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [578]:
len(words)

32033

In [579]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [580]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    # print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

In [581]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [582]:
# Embedding layer: lookup table
C = torch.randn((27, 2))
C.shape

torch.Size([27, 2])

In [583]:
# F.one_hot(torch.tensor(5), num_classes=27).float() @ C
# which is equal to
# c[5]

# Batch embedding lookup (output of embedding layer)
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [584]:
# Hidden layer

In [585]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [586]:
# 1. emb.view(-1, 6): Flatten (3, 2) into 6
# 2. @ W1 + b1: Use matrix multiplication to apply weights and add bias to 100
# neurons
# 3. torch.tanh(): Put all the number between [-1, 1] (Activation function)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [587]:
h

tensor([[-0.9523,  0.9996, -0.4235,  ..., -0.9542, -0.1544,  0.1623],
        [-0.9897,  0.9359,  0.6212,  ..., -0.5109, -0.9055, -0.3548],
        [ 0.2324,  0.3736,  0.9998,  ..., -0.8163, -0.9977, -0.2232],
        ...,
        [ 0.9363,  0.0450,  0.7309,  ...,  0.9915, -0.5479,  0.7896],
        [-0.3434,  0.6574, -0.5888,  ...,  0.1121,  0.1719,  0.9034],
        [-0.9716,  0.9972,  0.4407,  ...,  0.6543, -0.8969,  0.5120]])

In [588]:
h.shape

torch.Size([228146, 100])

In [589]:
# Output layer

In [590]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [591]:
logits = h @ W2 + b2

In [592]:
logits.shape

torch.Size([228146, 27])

In [593]:
counts = logits.exp() # 指数化，把结果变为正数

In [594]:
prob = counts / counts.sum(1, keepdims=True) # 归一化，把数字压缩到[0, 1]之间

In [595]:
prob.shape

torch.Size([228146, 27])

In [596]:
Y

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [597]:
loss = -prob[torch.arange(228146), Y].log().mean()
loss

tensor(16.1835)

In [598]:
# Put things together

In [599]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [600]:
# Parameter Initialization
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [601]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [602]:
for p in parameters:
  p.requires_grad = True

In [603]:
for _ in range(10):
  # mini-batch
  ix = torch.randint(0, X.shape[0], (32,))

  # Forward pass
  emb = C[X[ix]] # (32, 3, 2)
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)
  loss = F.cross_entropy(logits, Y[ix])

  # Backward pass
  for p in parameters:
    p.grad = None # 梯度清零
  loss.backward() # 自动求导

  # Update
  for p in parameters:
    p.data += -0.1 * p.grad


print(loss.item())

12.043460845947266


In [604]:
emb = C[X] # (32, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
logits = h @ W2 + b2 # (32, 27)
loss = F.cross_entropy(logits, Y)
loss

tensor(11.1543, grad_fn=<NllLossBackward0>)